# Actividad 7: Simulated Annealing para N-Queens

El problema de N-Reinas consiste en ubicar $N$ reinas en un tablero de $N \times N$ sin que dos reinas compartan fila, columna o diagonal.

Se representa un estado como una permutación: la posición `estado[fila]` indica la columna de la reina de esa fila. De esta forma, filas y columnas quedan resueltas automáticamente y el costo cuenta únicamente conflictos diagonales.

Simulated Annealing acepta ocasionalmente movimientos peores al comienzo para escapar de mínimos locales. La temperatura disminuye en cada iteración mediante `temperature *= cooling_rate`.

In [1]:
import math
import random


def costo(estado):
    """Cuenta pares de reinas que se atacan diagonalmente."""
    conflictos = 0
    for fila_1 in range(len(estado)):
        for fila_2 in range(fila_1 + 1, len(estado)):
            misma_diagonal = abs(estado[fila_1] - estado[fila_2]) == abs(fila_1 - fila_2)
            if misma_diagonal:
                conflictos += 1
    return conflictos


def estado_aleatorio(n):
    """Genera un estado con una reina por fila y por columna."""
    estado = list(range(n))
    random.shuffle(estado)
    return estado


def vecino(estado):
    """Genera un vecino intercambiando dos columnas."""
    nuevo_estado = estado.copy()
    fila_1, fila_2 = random.sample(range(len(estado)), 2)
    nuevo_estado[fila_1], nuevo_estado[fila_2] = nuevo_estado[fila_2], nuevo_estado[fila_1]
    return nuevo_estado


def grad_desc(n, max_steps=1000):
    """Búsqueda local de referencia mediante descenso de gradiente."""
    current = estado_aleatorio(n)
    current_cost = costo(current)

    for _ in range(max_steps):
        if current_cost == 0:
            break

        vecinos = [vecino(current) for _ in range(max(10, n * 2))]
        mejor_vecino = min(vecinos, key=costo)
        mejor_costo = costo(mejor_vecino)

        if mejor_costo >= current_cost:
            break

        current = mejor_vecino
        current_cost = mejor_costo

    return current, current_cost

In [2]:
def simulated_annealing(n, initial_temp, cooling_rate, max_steps=1000):
    """Resuelve N-Queens mediante Simulated Annealing.

    Devuelve exactamente (estado_final, costo_final).
    """
    if n < 1:
        raise ValueError('n debe ser mayor o igual que 1.')
    if initial_temp <= 0:
        raise ValueError('initial_temp debe ser positivo.')
    if not 0 < cooling_rate < 1:
        raise ValueError('cooling_rate debe estar entre 0 y 1.')

    current = estado_aleatorio(n)
    current_cost = costo(current)
    best = current.copy()
    best_cost = current_cost
    temperature = float(initial_temp)
    last_step = 0

    for step in range(1, max_steps + 1):
        last_step = step
        if current_cost == 0:
            break

        candidate = vecino(current)
        candidate_cost = costo(candidate)
        diferencia = candidate_cost - current_cost

        if diferencia <= 0 or random.random() < math.exp(-diferencia / temperature):
            current = candidate
            current_cost = candidate_cost

            if current_cost < best_cost:
                best = current.copy()
                best_cost = current_cost

        temperature *= cooling_rate
        if temperature < 1e-12:
            temperature = 1e-12

    return best, best_cost

In [4]:
# Cinco ejecuciones de prueba
n = 8
initial_temp = 10.0
cooling_rate = 0.995

resultados_sa = []
resultados_grad_desc = []

for semilla in range(5):
    random.seed(semilla)
    estado_sa, costo_sa = simulated_annealing(n, initial_temp, cooling_rate)
    resultados_sa.append(costo_sa)

    random.seed(semilla)
    estado_gd, costo_gd = grad_desc(n)
    resultados_grad_desc.append(costo_gd)

print(f'Parámetros de Simulated Annealing: T inicial={initial_temp}, cooling_rate={cooling_rate}, max_steps=1000 por defecto')
print(f'Costos de Simulated Annealing: {resultados_sa}')
print(f'Soluciones válidas de Simulated Annealing: {sum(costo == 0 for costo in resultados_sa)} de 5')
print(f'Costos de grad_desc: {resultados_grad_desc}')
print(f'Soluciones válidas de grad_desc: {sum(costo == 0 for costo in resultados_grad_desc)} de 5')

assert all(costo >= 0 for costo in resultados_sa)
assert all(len(simulated_annealing(1, initial_temp, cooling_rate)[0]) == 1 for _ in range(1))

Parámetros de Simulated Annealing: T inicial=10.0, cooling_rate=0.995, max_steps=1000 por defecto
Costos de Simulated Annealing: [0, 0, 0, 0, 0]
Soluciones válidas de Simulated Annealing: 5 de 5
Costos de grad_desc: [1, 2, 0, 1, 3]
Soluciones válidas de grad_desc: 1 de 5
